In [0]:
# """
# This Catalog is used for storing:
#     Metadata/configuration files in volumes and tables created out of those config files

# If multiple environments are used (e.g., dev/prod), this dynamically switches the catalog based on the selected environment.
# """

# # Environment selector
# dbutils.widgets.dropdown('env', 'dev', ['dev', 'prod'])

# # Creates environment-specific bronze catalog
# # Example:
# #   dev   -> dev_bronze
# #   prod  -> prod_bronze
# metadata_catalog = f"{dbutils.widgets.get('env')}_bronze"

### Imports

In [0]:
# ============================================================
# Python Standard Library Imports
# ============================================================
import re
from datetime import datetime
import json
import logging
import sys
import uuid
from collections import defaultdict
from delta.tables import DeltaTable
from zoneinfo import ZoneInfo

# ============================================================
# PySpark Core Imports
# ============================================================
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ============================================================
# PySpark SQL Functions
# ============================================================
from pyspark.sql.functions import (
    coalesce, col, collect_list, concat, current_timestamp,
    date_add, date_format, datediff, expr, lit,
    monotonically_increasing_id, regexp_replace, row_number,
    struct, substring, to_date, to_timestamp,
    trim, udf, when, year
)


# ============================================================
# PySpark Data Types
# ============================================================
from pyspark.sql.types import *

### Spark Config

In [0]:
spark.conf.set("spark.sql.ansi.enabled", "false")

### Run Notebooks
- user_defined_functions
- type_casting_functions
- helper_functions
- logger_functions
- nb_stg_scd

In [0]:
%run ./utils/nb_user_defined_functions

In [0]:
%run ./utils/nb_type_casting_functions

In [0]:
%run ./utils/nb_helper_functions

In [0]:
%run ./utils/nb_logger_functions

In [0]:
%run ./nb_stg_scd

### Col / Table Procesing Functions

In [0]:
def apply_typecasting_and_defaults(table_df, column_name, dest_data_type, dest_date_format=None, default_value=None):

    """
    Apply datatype casting and fill with default value if provided
    for a specific column based on destination datatype.

    Supported TypeCasting:
    - Numeric datatypes
    - Date
    - Timestamp
    - String
    """

    dtype = dest_data_type.lower()

    if dtype in NUMERIC_TYPES:
        return cast_numeric(
            table_df,
            column_name,
            NUMERIC_TYPES[dtype],
            default_value
        )

    elif dtype == "date":
        return transform_to_date(
            table_df,
            column_name,
            dest_date_format,
            default_value
        )

    elif dtype == "timestamp":
        return transform_to_timestamp(
            table_df,
            column_name,
            default_value
        )

    elif dtype == "string":
        return transform_to_string(
            table_df,
            column_name,
            default_value
        )

    return table_df

In [0]:
def apply_user_defined_functions(df, column_name, transformation_function, transformation_args):
    """
    Apply user-defined functions to a column.
    """
    ## Check if function is present in registry
    if transformation_function not in USER_DEFINED_TRANSFORMATION_FUNCTIONS_REGISTRY:
        logger.error("FAILED - Function not found in function registry !!")
        return
    
    ## Fetch the function
    transform_func = USER_DEFINED_TRANSFORMATION_FUNCTIONS_REGISTRY.get(transformation_function)

    ## Apply transformation function to the column
    if transformation_args is None:
        df = transform_func(df, column_name)
    else:
        df = transform_func(df, column_name, **transformation_args)

    return df

In [0]:
def apply_column_masking(masking_func_catalog, fully_qualified_sink_table_name,masking_func,cols):

    # -----------------------------------
    # Validate table exists
    # -----------------------------------
    if not spark.catalog.tableExists(fully_qualified_sink_table_name):
        raise Exception(
            f"Table does not exist: {fully_qualified_sink_table_name}"
        )

    # -----------------------------------
    # Parse catalog/schema/table
    # -----------------------------------
    catalog_name, schema_name, table_name = (
        fully_qualified_sink_table_name.lower().split(".")
    )

    # -----------------------------------
    # Apply masking
    # -----------------------------------
    for col in cols:
        sql_stmt = f"""
            ALTER TABLE {fully_qualified_sink_table_name}
            ALTER COLUMN {col}
            SET MASK {masking_func_catalog}.reference.{masking_func}
        """

        spark.sql(sql_stmt)

In [0]:
def apply_all_metadata(fully_qualified_sinktable_name, table_metadata, logger,columns_metadata=None):
    """
    Applies comments and tags to both the table and its columns in 
    """
    
    # ============================================================
    # TABLE COMMENTS & TAGS
    # ============================================================
    table_comments = table_metadata.get("table_description")
    table_tags = table_metadata.get("tags")

    # Apply table comment if it exists
    if table_comments:

        safe_table_comments = table_comments.replace("'", "''")
        spark.sql(f"""
            COMMENT ON TABLE {fully_qualified_sinktable_name} IS '{safe_table_comments}'
        """)

    # Apply table tags if they exist
    if table_tags:
        table_tag_pairs = process_tags(table_tags)
        spark.sql(f"""
            ALTER TABLE {fully_qualified_sinktable_name}
            SET TAGS ({table_tag_pairs})
        """)
        logger.info(f"Applied table tags to {fully_qualified_sinktable_name}")


    # ============================================================
    # COLUMN COMMENTS (BULK EXECUTION)
    # ============================================================
    comment_clauses = []
    for col_name, metadata in columns_metadata.items():

        col_description = metadata.get("description")

        if col_description:

            # Escape single quotes for SQL safety
            col_description = col_description.replace("'", "''")

            # ONLY include the identifier and the action clause
            clause = f"`{col_name}` COMMENT '{col_description}'"
            comment_clauses.append(clause)

    # Execute only if there are comments to update
    if comment_clauses:
        # Place ALTER COLUMN once, then join the specific attributes with a comma
        bulk_comment_sql = f"""
        ALTER TABLE {fully_qualified_sinktable_name}
        ALTER COLUMN {', '.join(comment_clauses)}
        """
        spark.sql(bulk_comment_sql)
        logger.info(f"Applied comments to {fully_qualified_sinktable_name}")

    # ============================================================
    # COLUMN TAGS (INDIVIDUAL EXECUTION)
    # ============================================================
    for col_name, metadata in columns_metadata.items():
        tags_string = metadata.get("tags")
        if tags_string:
            col_tag_pairs = process_tags(tags_string)
            
            spark.sql(f"""
                ALTER TABLE {fully_qualified_sinktable_name}
                ALTER COLUMN `{col_name}`
                SET TAGS ({col_tag_pairs})
            """)
            logger.info(f"Applied tags to column: `{col_name}`")


In [0]:
def apply_transformations(table_df, column_config_dict):
    """
    Apply transformations based on configuration dictionary.
    Optimized to avoid repeated DataFrame plan rebuilding.
    """

    ## Work on a single dataframe reference (no unnecessary recreation)
    df = table_df
    column_rename_mappings = {}

    for column_name in table_df.columns:

        try:
            column_name_lower = column_name.lower()

            if column_name_lower in column_config_dict:
                config = column_config_dict[column_name_lower]

                ## Get destination column configs
                dest_column_name = config.get("destination_column_name")
                dest_data_type = config.get("data_type")
                dest_date_format = config.get("date_format")
                default_value = config.get("default_value")

                ## Get transformation function and args if defined
                transformation_function = config.get("transformation_function")
                transformation_args = json.loads(
                    config.get("transformation_args") or "{}"
                )

                ## Apply  type casting and default value fill, if defined
                if dest_data_type:
                    df = apply_typecasting_and_defaults(
                        df,
                        column_name,
                        dest_data_type,
                        dest_date_format,
                        default_value
                    )

                ## Fill default value for columns with no typecasting is not defined
                if dest_data_type is None and default_value is not None:
                    df = fill_column_with_default_value(df, column_name_lower)

                ## If destination column is not provided then sanitize the column name
                if dest_column_name is None:
                    dest_column_name = sanitize_column_name(column_name)

                ## If transformation is defined, apply it            
                if transformation_function is not None:
                    df = apply_user_defined_functions(df, column_name, transformation_function, transformation_args)
                
                ## Add renamed/cleaned column name to column_rename_mappings
                column_rename_mappings[column_name] = dest_column_name

            else:
                # Sanitize column name if no config
                column_rename_mappings[column_name] = sanitize_column_name(column_name)

        except Exception as e:
            raise RuntimeError( f"Column Transformation Failed -> {column_name}") from e

    return df, column_rename_mappings

### Core Processing

In [0]:
def process_table(raw_table, stg_table,logger, column_configs=None, incremental_column=None,dest_incremental_column=None):
    """
    Main function to process raw table data and prepare it for staging.

    Steps performed:
    1. Construct fully qualified raw and staging table names.
    2. Support incremental loading if incremental column is provided.
    3. Read only new records from raw table.
    4. Apply column-level transformations if configs are provided.
    5. Sanitize and rename columns.
    6. Return transformed DataFrame.
    """

    raw_table = f"{raw_table['catalog']}.{raw_table['schema']}.{raw_table['table_name']}"
    stg_table = f"{stg_table['catalog']}.{stg_table['schema']}.{stg_table['table_name']}"
    column_rename_mapping = None
    new_record_count = 0
    load_type = None

    ## LOG Object
    table_log = {
        "raw_table": raw_table,
        "stg_table": stg_table,
        "table_processed_at": datetime.now(ZoneInfo("Asia/Kolkata"))
    }
    table_log["data"] = {}

    ## Check if incremental column is defined, to support incremental processing
    if isinstance(incremental_column, str):
        try:
            stg_df = read_table(stg_table)
            max_loaded = stg_df.agg(F.max(dest_incremental_column)).collect()[0][0]
            load_type = 'Incremental'
            
        except AnalysisException:
            ## If staging table doesn't exist yet, treat max_loaded_at as None (load all)
            load_type = 'Full Load'
            max_loaded = datetime(1700, 1, 1)

        try:
            ## Read only the latest data from raw table
            raw_df = read_table(raw_table).filter(
                F.col(incremental_column) > F.lit(max_loaded)
            )
            new_record_count = raw_df.count()
            logger.info(
                f"|___ Data Extraction Status : "
                f"Status: SUCCESS | "
                f"Extraction Type: {load_type} | "
                f"Max Loaded At: {max_loaded} | "
                f"{'No New Rows Found' if new_record_count == 0 else 'New Rows Identified'} | "
                f"Records: {new_record_count}"
            )

            table_log['status'] = 'Success'
            
            table_log["data"]["data_extraction"] = {
                "status": "Success",
                "extraction_method": load_type,
                "new_records": new_record_count
            }

        except Exception as e:
            logger.error(
                f"|___ Incremental Status : "
                f"Status: FAILED"
            )

            table_log['status'] = 'Failed'

            table_log["data"]["data_extraction"] = {
                "status": "Failed",
                "extraction_method": load_type,
                "new_records": new_record_count,
                "message": str(e)
            }

            return None, table_log

    else:
        raw_df = read_table(raw_table)
        new_record_count = raw_df.count()

    ## If no new rows return None
    if new_record_count == 0:
        logger.info("No new records to ingest - Process Completed")
        table_log['status'] = 'Success'

        table_log["data"]["data_extraction"] = {
                "status": "Success",
                "extraction_method": load_type,
                "new_records": new_record_count
            }
        return None, table_log   

    ## If column config exists apply those configurations
    if column_configs:
        column_config_dict = parse_column_configs(column_configs)
        try:
            raw_df, column_rename_mapping = apply_transformations(raw_df, column_config_dict) 

            table_log['status'] = 'Success'
            table_log["data"]["column_transformation"] = {
                "status": "Success"
            }

            logger.info(
                f"|___ Column Transformation: "
                f"Status: SUCCESS"
            )
        except Exception as e:

            table_log['status'] = 'Failed'

            table_log["data"]["column_transformation"] = {
                "status": "Failed",
                "message": str(e)
            }

            logger.error(
                f"|___ Column Transformation: "
                f"Status: FAILED | "
                f"Column Failed: {e} | "
                f"Error Message: {e}"
            )
            return None, table_log

    ## If no column configurations just sanitize all column names
    else:
        try:
            column_rename_mapping = sanitize_all_columns(raw_df)
            table_log['status'] = 'Success'

            table_log["data"]["column_transformation"] = {
                "status": "Success"
            }

        except Exception as e:
            table_log['status'] = 'Failed'

            table_log["data"]["column_transformation"] = {
                "status": "Failed",
                "message": str(e)
            }

            return None, table_log

    
    try:
        raw_df = rename_table_cols(raw_df, column_rename_mapping)
        table_log['status'] = 'Success'

        table_log["data"]["column_renaming"] = {
                "status": "Success"
            }

        logger.info(
                f"|___ Column Renaming: "
                f"Status: SUCCESS"
            )

    except Exception as e:
        table_log['status'] = 'Failed'

        table_log["data"]["column_renaming"] = {
                "status": "Failed",
                "message": str(e)
            }
        
        logger.error(
                f"|___ Column Renaming: "
                f"Status: FAILED | "
                f"Error Message: {e}"
            )
        return None,table_log
    
    return raw_df, table_log

In [0]:
def process_pipeline(source_table_obj,sink_table_obj, table_config,  incremental_column, dest_incremental_column, logger):
    """
    Processes tables with default transformations if schema_name and table_name are passed.
    Applies additional transformations based on config_grouped if the table is in config.
    """

    table_name = f"{source_table_obj['catalog']}.{source_table_obj['schema']}.{source_table_obj['table_name']}"

    if table_config:
        logger.info(
                f"|___ CONFIG: "
                f"Status: Found"
            )    
    else:
        logger.warning(
                f"|___ CONFIG: "
                f"Status: Not Found"
            )  

    ## Process Table using defined configuration
    return process_table(
            source_table_obj,
            sink_table_obj,
            logger,
            table_config.get("columns") if table_config else None,
            incremental_column,
            dest_incremental_column
            
        )        

In [0]:
def process_configured_tables(tables, config_object, metadata_catalog):

    ## Process Tables Defined In MetaData Table
    error_flag = 0

    PIPELINE_LOGS, notebook_run_id, notebook_start_time, job_id = get_pipeline_config()

    logger = get_logger()
    
    for table in tables:

        source_table_obj = {
            "catalog": table['source_catalog'],
            "schema": table['source_schema'],
            "table_name": table['source_table_name'],
        }

        sink_table_obj = {
            "catalog": table['sink_catalog'],
            "schema": table['sink_schema'],
            "table_name": table['sink_table_name'],
        }

        fully_qualified_src_table_name = f"{source_table_obj['catalog']}.{source_table_obj['schema']}.{source_table_obj['table_name']}"
        fully_qualified_sinktable_name = f"{sink_table_obj['catalog']}.{sink_table_obj['schema']}.{sink_table_obj['table_name']}"

        logger.info("=" * 80)
        logger.info(f"Started Processing Table : {fully_qualified_src_table_name}")

        # --------------------------------------------------------
        # Table And Column Config Extraction
        # --------------------------------------------------------

        ## Get configs associated with this table
        curr_table_config = {}
        config_rows = config_object.filter(
        (col("catalog") == source_table_obj["catalog"]) &
        (col("schema") == source_table_obj["schema"]) &
        (col("table_name") == source_table_obj["table_name"])
        ).collect()

        if config_rows:
            curr_table_config = config_rows[0].asDict()

        ## Get Column Rename Mappings
        col_name_mappings = get_col_mappings(fully_qualified_src_table_name, config_rows)

        ## Incremental Column - For incremntal load of the table
        ## Source Inc Column Name 
        incremental_column = table['incremental_column']

        ## Destination Inc Column Name
        dest_incremental_column = (
            col_name_mappings.get(table['incremental_column'])
            if col_name_mappings and table.get('incremental_column',{})
            else None
        )

        # --------------------------------------------------------
        # Table Processing
        # --------------------------------------------------------

        ## Process Table using defined config
        cleaned_df, table_log = process_pipeline(
            source_table_obj,
            sink_table_obj,
            curr_table_config,
            incremental_column,
            dest_incremental_column,
            logger
        )

        ## Check if table is enabled for SCD2
        is_scd_enabled = table['is_scd_enabled']

        primary_key_values = table['primary_key']
        incremental_column = dest_incremental_column

        # --------------------------------------------
        # Logs
        # ----------------------------------------------
        table_log['is_scd_enabled'] = True if is_scd_enabled else False
        table_log['primary_key'] = primary_key_values
        table_log['incremental_column'] = incremental_column
        table_log['notebook_run_id'] = notebook_run_id
        table_log['notebook_start_time'] = notebook_start_time
        table_log['pipeline_run_id'] = job_id


        ## If processsing the table gets failed skip the remaining steps
        if (table_log):

            if (table_log.get('status','Failed').lower() == "failed"):
                error_flag  = 1
                logger.error(f"Failed in processing table {fully_qualified_src_table_name}")
                PIPELINE_LOGS.append(table_log)
                continue
        else:
            error_flag  = 1
            logger.error(f"Failed in processing table {fully_qualified_src_table_name}")
            PIPELINE_LOGS.append(table_log)
            continue
            
        # --------------------------------------------------------
        # SCD / INC Load
        # --------------------------------------------------------

        ## PK Column for SCD / INC Load
        primary_key = (
            "|".join(
                col_name_mappings.get(p, p)
                for p in primary_key_values.split("|")
            )
            if col_name_mappings and primary_key_values else None
        )

        if cleaned_df is not None and is_scd_enabled :
            selected_columns = cleaned_df.columns
            try:
                # This function must exist in imported notebook
                process_scd_table(
                    table,
                    cleaned_df,
                    selected_columns,
                    incremental_column,
                    primary_key
                )

                table_log['status'] = 'Success'

                table_log["data"]["load_strategy"] = {
                    "status": "Success",
                    "insertion_type": "SCD2"
                }

            except Exception as e:
                table_log['status'] = 'Failed'

                table_log["data"]["load_strategy"] = {
                    "status": "Failed",
                    "insertion_type": "SCD2",
                    "message": str(e)
                }

        ## Full Load
        elif cleaned_df is not None and table_log.get('data', {}).get('load', {}).get('load_type') != 'Incremental':
            try:
                write_table(
                    cleaned_df,
                    fully_qualified_sinktable_name,
                )

                table_log['status'] = 'Success'

                table_log["data"]["load_strategy"] = {
                    "status": "Success",
                    "insertion_type": "Full Load"
                }

            except Exception as e:
                table_log['status'] = 'Failed'

                table_log["data"]["load_strategy"] = {
                    "status": "Failed",
                    "insertion_type": "Full Load",
                    "message": str(e)
                }

        ## Only Incremental Load, No SCD
        elif cleaned_df is not None:
            try:
                ## If INC load is defined but PK is not provided throw error
                if primary_key is None or len(primary_key.strip()) == 0:
                    raise ValueError(
                        f"Primary key is missing for table: {fully_qualified_src_table_name}"
                    )   
                    
                merge_upsert(
                    cleaned_df,
                    fully_qualified_sinktable_name,
                    primary_key

                )

                table_log['status'] = 'Success'

                table_log["data"]["load_strategy"] = {
                    "status": "Success",
                    "insertion_type": "Incremental"
                }

            except Exception as e:
                table_log['status'] = 'Failed'

                table_log["data"]["load_strategy"] = {
                    "status": "Failed",
                    "insertion_type": "Incremental",
                    "message": str(e)
                }

        else:
            logger.info(f"No New Data To Load")

            table_log['status'] = 'Success'

            table_log["data"]["load_strategy"] = {
                    "status": "Success",
                    "insertion_type": "",
                    "message": "No New Data To Load"
                }

            
        ## If writing the table gets failed skip the remaining steps
        if (table_log and table_log.get('status','Failed').lower() == "failed"):
            error_flag  = 1
            logger.error(f"Failed in writing table {fully_qualified_sinktable_name}")
            PIPELINE_LOGS.append(table_log)
            continue
        
        # --------------------------------------------------------
        # Column and Table Metadata
        # --------------------------------------------------------
        try:
            ## Column Description Extraction
            column_description_mapping = {

            col_name_mappings.get(
                col.column_name,
                col.column_name
            ): {
                "description":col.column_description,
                "tags": col.tags,
                "masking":col.column_masking
            }

            for col in curr_table_config.get("columns",[])

            if col and (col.column_description is not None or col.tags is not None)

            }

            apply_all_metadata(fully_qualified_sinktable_name,table,logger,column_description_mapping)
            table_log['status'] = 'Success'
            table_log["data"]["metadata"] = {
                    "status": "Success"
                }
            logger.info(
                f"|___ Column Metadata: "
                f"Status: SUCCESS"
            )

        except Exception as e:
            table_log['status'] = 'Failed'
            table_log["data"]["metadata"] = {
                    "status": "Failed",
                    "message": str(e)
                }
            logger.info(
                f"|___ Column Renaming: "
                f"Status: FAILED"
            )

        # --------------------------------------------------------
        # Column Masking
        # --------------------------------------------------------
        try:

            ## column masking start
            masking_dict = defaultdict(list)
            for col_config in curr_table_config.get("columns", []):
                masking_func = col_config["column_masking"]
                if masking_func:
                    destination_col = col_name_mappings[col_config["column_name"]]
                    masking_dict[masking_func].append(destination_col)

            masking_dict = dict(masking_dict)

            ## Apply Column Masking
            for masking_func,mask_cols in masking_dict.items():
                apply_column_masking(metadata_catalog,fully_qualified_sinktable_name,masking_func,mask_cols)
            

            table_log["status"] = "Success"
            # table_log["data"] += f"\ncolumn_masking -> Success"
            table_log["data"]["column_masking"] = {
                    "status": "Success"
                }
            logger.info(
                f"|___ Column Masking: "
                f"Status: SUCCESS"
            )

        except Exception as e:
            table_log["status"] = "Failed"
            table_log["data"]["column_masking"] = {
                    "status": "Failed",
                    "message": str(e)
                }
                
            logger.info(
                f"|___ Column Masking: "
                f"Status: FAILED"
            )

        if (table_log and table_log.get("status","Failed").lower() == "failed"):
            error_flag  = 1
            logger.error(f"Failed in updating table {fully_qualified_sinktable_name}")
            PIPELINE_LOGS.append(table_log)
            continue

        ## Append Logs
        PIPELINE_LOGS.append(table_log)

        logger.info(f"Completed Processing Table : {fully_qualified_src_table_name}")

    return PIPELINE_LOGS,error_flag
